In [54]:
import requests
import os
import re
from datetime import datetime

def sanitize_filename(title):
    return re.sub(r'[^a-zA-Z0-9\-]', '', title.replace(' ', '-')).lower()

def fetch_metadata_from_doi(doi):
    url = f"https://api.crossref.org/works/{doi}"
    response = requests.get(url)
    response.raise_for_status()
    # print(response.json()['message'])
    return response.json()['message']

def generate_markdown(metadata, paper_url):
    # Extract key info
    title = metadata.get('title', [''])[0]
    authors = metadata.get('author', [])
    author_str = ', '.join([f"{a['family']}, {a['given'][0]}." for a in authors])
    journal = metadata.get('container-title', [''])[0]
    # pub_date_parts = metadata.get('published-print', {}).get('date-parts', [[2025, 1, 1]])[0]
    pub_date_parts = metadata.get('indexed', {}).get('date-parts', [[2025, 1, 1]])[0]
    pub_date = datetime(*pub_date_parts)
    date_str = pub_date.strftime('%d/%m/%Y')
    permalink_date = pub_date.strftime('%d/%m/%Y')
    citation = f"{author_str}, {pub_date.year}. {title}. {journal}."

    # Filename-safe slug
    # slug = sanitize_filename(title)
    slug = sanitize_filename(title).split('-')[0]
    filename = f"{permalink_date.replace('/', '-')}-{slug}.md"

    # Markdown content
    content = f"""---
title: "{title}"
collection: publications
permalink: /publication/{permalink_date.replace('/', '/')}-post-{slug}
excerpt: '{title}'
date: {date_str}
venue: '{journal}'
paperurl: '{paper_url}'
citation: '{citation} https://doi.org/{metadata.get("DOI")}'
---

{title}

[Download paper here]({paper_url})

Recommended citation: {citation} https://doi.org/{metadata.get("DOI")}
"""

    return filename, content


In [55]:
# title = metadata.get('title', [''])[0]
# pub_date_parts = metadata.get('indexed', {}).get('date-parts', [[2025, 1, 1]])[0]
# pub_date = datetime(*pub_date_parts)
# date_str = pub_date.strftime('%d/%m/%Y')
# permalink_date = pub_date.strftime('%d/%m/%Y')
# print(permalink_date)


# slug = sanitize_filename(title).split('-')[0]
# print(f"Slug: {slug}")
# filename = f"{permalink_date.replace('/', '-')}-{slug}.md"
# print(f"Filename: {filename}")

In [56]:
# doi_input = "https://doi.org/10.1109/JSTARS.2025.3581499"  # Replace with your actual DOI
# paper_url = "https://ieeexplore.ieee.org/abstract/document/11045930"  # Update as needed
# metadata = fetch_metadata_from_doi(doi_input)
# # pub_date_parts = metadata.get('published-print', {}).get('date-parts', [[2025, 1, 1]])[0]
# # print(pub_date_parts)
# from pprint import pprint
# pprint(metadata)

In [57]:
doi_input = "10.1016/j.mex.2025.103458"  # Replace with your actual DOI
paper_url = "https://www.sciencedirect.com/science/article/pii/S2215016125003036"  # Update as needed

try:
    metadata = fetch_metadata_from_doi(doi_input)
    filename, markdown = generate_markdown(metadata, paper_url)
    
    output_dir = "./_publications"
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, filename), 'w', encoding='utf-8') as f:
        f.write(markdown)

    print(f"✅ Markdown file generated: {filename}")

except Exception as e:
    print(f"❌ Error: {e}")

✅ Markdown file generated: 15-07-2025-design.md
